<a href="https://colab.research.google.com/github/ishreya-dev/tinystories-gpt/blob/v3-transformer/slm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install datasets

In [2]:
!pip install -q transformers datasets

In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F

from datasets import load_dataset

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

True
Tesla T4


In [4]:
import re
from collections import Counter

In [5]:
from datasets import load_dataset

dataset = load_dataset("roneneldan/TinyStories")

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

data/train-00001-of-00004-5852b56a2bd28f(…): downloading bytes:           |  0.00B            

...

data/train-00003-of-00004-d243063613e5a0(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-869c898b5(…): reconstructing file:   0%|          |  0.00B / 9.99MB            

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [6]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})


In [7]:
print(dataset["train"][0])

{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


In [8]:
dataset["train"][0]["text"]

'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'

In [9]:
print("Number of training examples:", len(dataset["train"]))
print("Number of validation examples:", len(dataset["validation"]))

print("\nColumns:")
print(dataset["train"].column_names)

print("\nFirst story:")
print(dataset["train"][0]["text"])

Number of training examples: 2119719
Number of validation examples: 21990

Columns:
['text']

First story:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


In [10]:
for i in range(3):
  print(f"\n--- Story {i+1} ---")
  print(dataset["train"][i]["text"][:300])


--- Story 1 ---
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and 

--- Story 2 ---
Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.

One day, Beep was driving in the park when he saw a big tree. The tree had many leaves that were falling. Bee

--- Story 3 ---
One day, a little fish named Fin was swimming near the shore. He saw a big crab and wanted to be friends. "Hi, I am Fin. Do you want to play?" asked the little fish. The crab looked at Fin and said, "No, I don't want to play. I am cold and I don't feel fine."

Fin felt sad but wanted to help the cra


In [11]:
# Take a small portion of the training data first
text = "\n".join(dataset["train"][:1000]["text"])

print(text[:500])
print("\nTotal characters:", len(text))

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them b

Total characters: 942639


In [12]:
# Find all unique characters
chars = sorted(list(set(text)))

# Vocabulary size
vocab_size = len(chars)

print("Vocabulary size:", vocab_size)
print(chars)

Vocabulary size: 75
['\n', ' ', '!', '"', '$', "'", ',', '-', '.', '0', '1', '2', '3', '8', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'â', 'œ', '“', '”', '€', '™']


In [13]:
# String → Integer
stoi = {ch: i for i, ch in enumerate(chars)}

# Integer → String
itos = {i: ch for i, ch in enumerate(chars)}

print(stoi)

{'\n': 0, ' ': 1, '!': 2, '"': 3, '$': 4, "'": 5, ',': 6, '-': 7, '.': 8, '0': 9, '1': 10, '2': 11, '3': 12, '8': 13, ':': 14, ';': 15, '?': 16, 'A': 17, 'B': 18, 'C': 19, 'D': 20, 'E': 21, 'F': 22, 'G': 23, 'H': 24, 'I': 25, 'J': 26, 'K': 27, 'L': 28, 'M': 29, 'N': 30, 'O': 31, 'P': 32, 'Q': 33, 'R': 34, 'S': 35, 'T': 36, 'U': 37, 'V': 38, 'W': 39, 'X': 40, 'Y': 41, 'Z': 42, 'a': 43, 'b': 44, 'c': 45, 'd': 46, 'e': 47, 'f': 48, 'g': 49, 'h': 50, 'i': 51, 'j': 52, 'k': 53, 'l': 54, 'm': 55, 'n': 56, 'o': 57, 'p': 58, 'q': 59, 'r': 60, 's': 61, 't': 62, 'u': 63, 'v': 64, 'w': 65, 'x': 66, 'y': 67, 'z': 68, 'â': 69, 'œ': 70, '“': 71, '”': 72, '€': 73, '™': 74}


Character level tokenizer

In [14]:
def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

In [15]:
example = "Hello Lily!"

encoded = encode(example)

print("Original:", example)
print("Encoded:", encoded)
print("Decoded:", decode(encoded))

Original: Hello Lily!
Encoded: [24, 47, 54, 54, 57, 1, 28, 51, 54, 67, 2]
Decoded: Hello Lily!


In [16]:
# Convert our entire text into token IDs
data = encode(text)

print(data[:20])
print("Total tokens:", len(data))

[31, 56, 47, 1, 46, 43, 67, 6, 1, 43, 1, 54, 51, 62, 62, 54, 47, 1, 49, 51]
Total tokens: 942639


In [17]:
example_data = data[:20]

x = example_data[:-1]
y = example_data[1:]

print("Input IDs: ", x)
print("Target IDs:", y)

print("\nInput text: ", decode(x))
print("Target text:", decode(y))

Input IDs:  [31, 56, 47, 1, 46, 43, 67, 6, 1, 43, 1, 54, 51, 62, 62, 54, 47, 1, 49]
Target IDs: [56, 47, 1, 46, 43, 67, 6, 1, 43, 1, 54, 51, 62, 62, 54, 47, 1, 49, 51]

Input text:  One day, a little g
Target text: ne day, a little gi


In [18]:
data = torch.tensor(encode(text), dtype=torch.long)

print(data[:20])
print(type(data))
print(data.dtype)

tensor([31, 56, 47,  1, 46, 43, 67,  6,  1, 43,  1, 54, 51, 62, 62, 54, 47,  1,
        49, 51])
<class 'torch.Tensor'>
torch.int64


BPETokenizer

In [19]:
class BPETokenizer:

    def __init__(self):
        # Start with an empty vocabulary
        self.vocab = {}

        # Store learned merge rules
        self.merges = {}

    def pre_tokenize(self, text):
        """
        Split text into words, spaces, and punctuation.
        """
        pattern = r"\w+|[^\w\s]|\s+"

        return re.findall(pattern, text)

    def get_pair_counts(self, sequences):
        """
        Count how often adjacent token pairs occur.
        """
        pair_counts = Counter()

        for sequence in sequences:

            for i in range(len(sequence) - 1):

                pair = (
                    sequence[i],
                    sequence[i + 1]
                )

                pair_counts[pair] += 1

        return pair_counts

    def merge_pair(self, sequence, pair):
        """
        Merge a specific adjacent pair into one token.
        """

        new_sequence = []

        i = 0

        while i < len(sequence):

            # Check whether the current token and next token
            # match the pair we want to merge
            if (
                i < len(sequence) - 1
                and sequence[i] == pair[0]
                and sequence[i + 1] == pair[1]
            ):

                # Merge the two tokens
                new_sequence.append(
                    sequence[i] + sequence[i + 1]
                )

                # Skip both tokens
                i += 2

            else:
                new_sequence.append(sequence[i])
                i += 1

        return new_sequence

    def train(self, text, vocab_size):

        # Split text into pieces using regex
        pieces = self.pre_tokenize(text)

        # Convert each piece into characters
        sequences = [
            list(piece)
            for piece in pieces
        ]

        # Get all unique starting characters
        base_vocab = sorted(
            set(char for piece in pieces for char in piece)
        )

        # Start vocabulary with individual characters
        self.vocab = {
            i: char
            for i, char in enumerate(base_vocab)
        }

        # Start assigning new IDs after base vocabulary
        next_id = len(self.vocab)

        print("Initial vocabulary size:", len(self.vocab))

        # Keep learning merges until target vocabulary size
        while len(self.vocab) < vocab_size:

            # Count all adjacent pairs
            pair_counts = self.get_pair_counts(sequences)

            # Stop if no pairs are left
            if not pair_counts:
                break

            # Find the most frequent pair
            best_pair = max(
                pair_counts,
                key=pair_counts.get
            )

            # Create the merged token
            merged_token = (
                best_pair[0] + best_pair[1]
            )

            # Store the merge rule
            self.merges[best_pair] = merged_token

            # Apply this merge to every sequence
            sequences = [
                self.merge_pair(sequence, best_pair)
                for sequence in sequences
            ]

            # Add merged token to vocabulary
            self.vocab[next_id] = merged_token

            next_id += 1

            # Show progress occasionally
            if len(self.vocab) % 100 == 0:
                print(
                    f"Vocabulary size: {len(self.vocab)} | "
                    f"Latest merge: {best_pair} -> {merged_token}"
                )

        print("\nFinal vocabulary size:", len(self.vocab))

    def encode(self, text):

        # Split text into pieces
        pieces = self.pre_tokenize(text)

        # Store final token IDs
        token_ids = []

        # Create reverse vocabulary
        token_to_id = {
            token: token_id
            for token_id, token in self.vocab.items()
        }

        for piece in pieces:

            # Start with individual characters
            sequence = list(piece)

            # Apply all learned BPE merges
            for pair in self.merges:

                sequence = self.merge_pair(
                    sequence,
                    pair
                )

            # Convert to IDs AFTER all merges are finished
            for token in sequence:

                token_ids.append(
                    token_to_id[token]
                )

        return token_ids

    def decode(self, token_ids):
      # Convert each token ID back to its token
      tokens = [
          self.vocab[token_id]
          for token_id in token_ids
          ]

      # Join all tokens back into text
      text = "".join(tokens)

      return text

In [20]:
tokenizer = BPETokenizer()

tokenizer.train(
    text,
    vocab_size=1024
)

Initial vocabulary size: 75
Vocabulary size: 100 | Latest merge: ('i', 'd') -> id
Vocabulary size: 200 | Latest merge: ('ri', 'end') -> riend
Vocabulary size: 300 | Latest merge: ('Tim', 'my') -> Timmy
Vocabulary size: 400 | Latest merge: ('d', 'own') -> down
Vocabulary size: 500 | Latest merge: ('man', 'y') -> many
Vocabulary size: 600 | Latest merge: ('pla', 'ce') -> place
Vocabulary size: 700 | Latest merge: ('go', 'ing') -> going
Vocabulary size: 800 | Latest merge: ('do', 'es') -> does
Vocabulary size: 900 | Latest merge: ('but', 'ter') -> butter
Vocabulary size: 1000 | Latest merge: ('we', 'l') -> wel

Final vocabulary size: 1024


In [21]:
# Show some learned vocabulary tokens

print("First 20 vocabulary entries:\n")

for token_id, token in list(tokenizer.vocab.items())[:20]:
    print(token_id, "->", repr(token))


print("\nSome learned BPE tokens:\n")

for token_id, token in list(tokenizer.vocab.items())[75:100]:
    print(token_id, "->", repr(token))

First 20 vocabulary entries:

0 -> '\n'
1 -> ' '
2 -> '!'
3 -> '"'
4 -> '$'
5 -> "'"
6 -> ','
7 -> '-'
8 -> '.'
9 -> '0'
10 -> '1'
11 -> '2'
12 -> '3'
13 -> '8'
14 -> ':'
15 -> ';'
16 -> '?'
17 -> 'A'
18 -> 'B'
19 -> 'C'

Some learned BPE tokens:

75 -> 'he'
76 -> 'an'
77 -> 'the'
78 -> 'ed'
79 -> 'and'
80 -> 'to'
81 -> 'in'
82 -> 're'
83 -> 'ou'
84 -> 'it'
85 -> 'wa'
86 -> 'ha'
87 -> 'er'
88 -> 'en'
89 -> '\n\n'
90 -> 'The'
91 -> 'was'
92 -> 'on'
93 -> 'om'
94 -> 'ar'
95 -> 'is'
96 -> 'ing'
97 -> 'sa'
98 -> 'il'
99 -> 'id'


In [22]:
sample_text = "One day, the little girl was happy."

encoded = tokenizer.encode(sample_text)
decoded = tokenizer.decode(encoded)

print("ORIGINAL:")
print(repr(sample_text))

print("\nDECODED:")
print(repr(decoded))

print("\nOriginal length:", len(sample_text))
print("Decoded length:", len(decoded))

print("\nEncoded IDs:")
print(encoded)

print("\nCharacter-by-character comparison:")

for i, (original_char, decoded_char) in enumerate(
    zip(sample_text, decoded)
):
    if original_char != decoded_char:
        print(
            f"First difference at position {i}: "
            f"original={repr(original_char)}, "
            f"decoded={repr(decoded_char)}"
        )
        break
else:
    if len(sample_text) != len(decoded):
        print("Text content matches up to the shorter length.")
    else:
        print("No differences found.")

print("\nMATCH:", sample_text == decoded)

ORIGINAL:
'One day, the little girl was happy.'

DECODED:
'One day, the little girl was happy.'

Original length: 35
Decoded length: 35

Encoded IDs:
[217, 1, 152, 6, 1, 77, 1, 195, 1, 225, 1, 91, 1, 203, 8]

Character-by-character comparison:
No differences found.

MATCH: True


In [23]:
# Encode the full text using our trained BPE tokenizer

data = torch.tensor(
    tokenizer.encode(text),
    dtype=torch.long
)

print("Total BPE tokens:", len(data))
print("Vocabulary size:", len(tokenizer.vocab))

Total BPE tokens: 459429
Vocabulary size: 1024


In [24]:
# 90% for training, 10% for validation

n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print("Training tokens:", len(train_data))
print("Validation tokens:", len(val_data))

Training tokens: 413486
Validation tokens: 45943


In [25]:
batch_size = 32
block_size = 128   # V2 change

n_embd = 128      # size of each embedding vector
n_head = 4        # number of attention heads
n_layer = 4       # number of Transformer blocks

dropout = 0.1

device = "cuda" if torch.cuda.is_available() else "cpu"

In [26]:
token_embedding_table = nn.Embedding(
    vocab_size,
    n_embd
).to(device)

In [27]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [28]:
def get_batch(split):

    data_split = train_data if split == "train" else val_data

    ix = torch.randint(
        len(data_split) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        data_split[i:i + block_size]
        for i in ix
    ])

    y = torch.stack([
        data_split[i + 1:i + block_size + 1]
        for i in ix
    ])

    return x.to(device), y.to(device)

In [29]:
vocab_size = len(tokenizer.vocab)

print("Vocabulary size:", vocab_size)
print("Data min:", data.min().item())
print("Data max:", data.max().item())

Vocabulary size: 1024
Data min: 0
Data max: 1023


In [30]:
test = torch.tensor([0, 1, 2, 1023])

print(test.to(device))

tensor([   0,    1,    2, 1023], device='cuda:0')


In [31]:
xb, yb = get_batch("train")

print("Input shape:", xb.shape)
print("Target shape:", yb.shape)

print(
    tokenizer.decode(
        xb[0].cpu().tolist()
    )
)

Input shape: torch.Size([32, 128])
Target shape: torch.Size([32, 128])
ves.

So Pixie decided to solve the mystery. She took out a magnifying glass and looked very carefully. After a long time of looking, Pixie finally saw what it was - a beautiful butterfly!

Pixie was so happy to have solved the mystery. She waved goodbye to the 


In [32]:
print("Vocabulary size:", len(tokenizer.vocab))

print("Minimum token ID:", min(data.tolist()))
print("Maximum token ID:", max(data.tolist()))

print("Any invalid IDs:",
      (data >= len(tokenizer.vocab)).any().item())

Vocabulary size: 1024
Minimum token ID: 0
Maximum token ID: 1023
Any invalid IDs: False


In [33]:
sample_ids = data[:50].tolist()

print(tokenizer.decode(sample_ids))

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it


In [35]:
print("INPUT:")

print(
    tokenizer.decode(
        xb[0].cpu().tolist()
    )
)

print("\nTARGET:")

print(
    tokenizer.decode(
        yb[0].cpu().tolist()
    )
)

INPUT:
ves.

So Pixie decided to solve the mystery. She took out a magnifying glass and looked very carefully. After a long time of looking, Pixie finally saw what it was - a beautiful butterfly!

Pixie was so happy to have solved the mystery. She waved goodbye to the 

TARGET:
s.

So Pixie decided to solve the mystery. She took out a magnifying glass and looked very carefully. After a long time of looking, Pixie finally saw what it was - a beautiful butterfly!

Pixie was so happy to have solved the mystery. She waved goodbye to the butter


In [36]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocab_size,
            vocab_size
        )

    def forward(self, idx, targets=None):

        # Input BPE token IDs -> vocabulary logits
        logits = self.token_embedding_table(idx)

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss


    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            # Get predictions
            logits, _ = self(idx)

            # Take prediction for the last token
            logits = logits[:, -1, :]

            # Convert scores into probabilities
            probs = F.softmax(
                logits,
                dim=-1
            )

            # Sample the next BPE token
            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )

            # Add the new token
            idx = torch.cat(
                (idx, idx_next),
                dim=1
            )

        return idx

In [37]:
model = BigramLanguageModel(vocab_size)
model = model.to(device)

logits, loss = model(xb, yb)

print("Logits shape:", logits.shape)
print("Loss:", loss)

Logits shape: torch.Size([4096, 1024])
Loss: tensor(7.4394, device='cuda:0', grad_fn=<NllLossBackward0>)


In [38]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

In [39]:
max_iters = 5000

for step in range(max_iters):

    # Get a random batch
    xb, yb = get_batch("train")

    # Forward pass
    logits, loss = model(xb, yb)

    # Remove gradients from the previous step
    optimizer.zero_grad()

    # Calculate gradients
    loss.backward()

    # Update model parameters
    optimizer.step()

    # Print loss every 500 steps
    if step % 500 == 0:
        print(f"Step {step}: Loss = {loss.item():.4f}")

Step 0: Loss = 7.4365
Step 500: Loss = 6.7836
Step 1000: Loss = 6.1228
Step 1500: Loss = 5.5437
Step 2000: Loss = 5.0701
Step 2500: Loss = 4.5781
Step 3000: Loss = 4.1980
Step 3500: Loss = 3.9017
Step 4000: Loss = 3.6174
Step 4500: Loss = 3.4450


In [41]:
def generate(self, idx, max_new_tokens):

    for _ in range(max_new_tokens):

        # Get predictions
        logits, loss = self(idx)

        # Take logits from the last position
        logits = logits[:, -1, :]

        # Convert logits into probabilities
        probs = F.softmax(logits, dim=-1)

        # Sample the next token
        idx_next = torch.multinomial(probs, num_samples=1)

        # Add the predicted token
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [43]:
context = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
)

generated_ids = model.generate(
    context,
    max_new_tokens=200
)[0].cpu().tolist()

print(
    tokenizer.decode(generated_ids)
)


esserbeautiful take But courtoongs. the ler! something that go.

TheyhenearYesgottyfelt theOkno lve Lily deciddLily playunus.

Onearound playing and couldnbecamequily veryainguadventhaira Thenlearned day, the The atcharound askwent and was athing dad'sshowed chohurtflerapped seen Everyonetoo flowersughtAfter âbyefightersleaddensee to He of upontrangryant is there Thefieventhroughet, funman yesickhaving carcookiesddll?" felt lived Ice, up ney beinglittle his day, new went and Timfelbecamepationlaughedill 


In [44]:
# Positional embedding table
position_embedding_table = nn.Embedding(
    block_size,
    n_embd
).to(device)

# Positions: 0, 1, 2, ..., 63
positions = torch.arange(block_size, device=device)

# Get embedding for each position
position_embeddings = position_embedding_table(positions)

print("Position shape:", positions.shape)
print("Position embedding shape:", position_embeddings.shape)

Position shape: torch.Size([128])
Position embedding shape: torch.Size([128, 128])


In [46]:
class Head(nn.Module):
    """
    One head of causal self-attention.
    """

    def __init__(self, head_size):

        super().__init__()

        self.key = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.query = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        self.value = nn.Linear(
            n_embd,
            head_size,
            bias=False
        )

        # Causal mask
        self.register_buffer(
            "tril",
            torch.tril(
                torch.ones(
                    block_size,
                    block_size
                )
            )
        )

        self.dropout = nn.Dropout(dropout)


    def forward(self, x):

        B, T, C = x.shape

        # Keys and queries
        k = self.key(x)
        q = self.query(x)

        # Attention scores
        wei = q @ k.transpose(-2, -1)

        # Scale scores using head dimension
        wei = wei * (k.size(-1) ** -0.5)

        # Prevent attending to future tokens
        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        # Convert scores to probabilities
        wei = F.softmax(
            wei,
            dim=-1
        )

        wei = self.dropout(wei)

        # Values
        v = self.value(x)

        # Weighted aggregation
        out = wei @ v

        return out

In [49]:
# Create one attention head
head_size = n_embd // n_head

head = Head(head_size).to(device)

# Get a batch
xb, yb = get_batch("train")

# Convert token IDs into embeddings
token_embedding_table = nn.Embedding(
    vocab_size,
    n_embd
).to(device)

position_embedding_table = nn.Embedding(
    block_size,
    n_embd
).to(device)

B, T = xb.shape

tok_emb = token_embedding_table(xb)

pos_emb = position_embedding_table(
    torch.arange(T, device=device)
)

x = tok_emb + pos_emb

# Pass through attention head
out = head(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)

Input shape: torch.Size([32, 128, 128])
Output shape: torch.Size([32, 128, 32])


In [51]:
class MultiHeadAttention(nn.Module):
    """
    Multiple attention heads running in parallel.
    """

    def __init__(self, num_heads, head_size):

        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size)
            for _ in range(num_heads)
        ])

        self.proj = nn.Linear(
            num_heads * head_size,
            n_embd
        )

        self.dropout = nn.Dropout(dropout)


    def forward(self, x):

        # Run all attention heads
        out = torch.cat(
            [head(x) for head in self.heads],
            dim=-1
        )

        # Mix information from all heads
        out = self.proj(out)

        out = self.dropout(out)

        return out

In [52]:
multihead = MultiHeadAttention(
    num_heads=n_head,
    head_size=n_embd // n_head
).to(device)

out = multihead(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)

Input shape: torch.Size([32, 128, 128])
Output shape: torch.Size([32, 128, 128])


In [53]:
class FeedForward(nn.Module):
    """
    A simple feed-forward neural network.
    """

    def __init__(self, n_embd):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                n_embd,
                4 * n_embd
            ),

            nn.ReLU(),

            nn.Linear(
                4 * n_embd,
                n_embd
            ),

            nn.Dropout(dropout)
        )

    def forward(self, x):

        return self.net(x)

In [54]:
ffwd = FeedForward(n_embd).to(device)

out = ffwd(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)

Input shape: torch.Size([32, 128, 128])
Output shape: torch.Size([32, 128, 128])


In [55]:
ffwd = FeedForward(n_embd).to(device)

out = ffwd(x)

print("Output shape:", out.shape)

Output shape: torch.Size([32, 128, 128])


In [56]:
class Block(nn.Module):
    """
    Transformer block:
    communication followed by computation.
    """

    def __init__(self, n_embd, n_head):

        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(
            n_head,
            head_size
        )

        self.ffwd = FeedForward(
            n_embd
        )

        self.ln1 = nn.LayerNorm(
            n_embd
        )

        self.ln2 = nn.LayerNorm(
            n_embd
        )


    def forward(self, x):

        # Self-attention with residual connection
        x = x + self.sa(
            self.ln1(x)
        )

        # FeedForward with residual connection
        x = x + self.ffwd(
            self.ln2(x)
        )

        return x

In [57]:
block = Block(
    n_embd,
    n_head
).to(device)

out = block(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)

Input shape: torch.Size([32, 128, 128])
Output shape: torch.Size([32, 128, 128])


In [58]:
class TransformerLanguageModel(nn.Module):

    def __init__(self):

        super().__init__()

        # Convert BPE token IDs into vectors
        self.token_embedding_table = nn.Embedding(
            vocab_size,
            n_embd
        )

        # Learn positional information
        self.position_embedding_table = nn.Embedding(
            block_size,
            n_embd
        )

        # Stack multiple Transformer blocks
        self.blocks = nn.Sequential(
            *[
                Block(
                    n_embd,
                    n_head
                )
                for _ in range(n_layer)
            ]
        )

        # Final normalization
        self.ln_f = nn.LayerNorm(
            n_embd
        )

        # Convert embeddings into vocabulary logits
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size
        )


    def forward(self, idx, targets=None):

        B, T = idx.shape

        # Token embeddings
        tok_emb = self.token_embedding_table(idx)

        # Position embeddings
        pos_emb = self.position_embedding_table(
            torch.arange(
                T,
                device=idx.device
            )
        )

        # Combine token and position information
        x = tok_emb + pos_emb

        # Pass through 4 Transformer blocks
        x = self.blocks(x)

        # Final normalization
        x = self.ln_f(x)

        # Vocabulary prediction
        logits = self.lm_head(x)

        loss = None

        if targets is not None:

            B, T, C = logits.shape

            logits = logits.view(
                B * T,
                C
            )

            targets = targets.view(
                B * T
            )

            loss = F.cross_entropy(
                logits,
                targets
            )

        return logits, loss

In [59]:
model = TransformerLanguageModel().to(device)

logits, loss = model(xb, yb)

print("Logits shape:", logits.shape)
print("Loss:", loss)

Logits shape: torch.Size([4096, 1024])
Loss: tensor(7.1340, device='cuda:0', grad_fn=<NllLossBackward0>)


In [61]:
num_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Total parameters: {num_params:,}")

Total parameters: 1,071,360


In [62]:
@torch.no_grad()
def estimate_loss():

    out = {}

    # Switch model to evaluation mode
    model.eval()

    for split in ["train", "val"]:

        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):

            xb, yb = get_batch(split)

            _, loss = model(xb, yb)

            losses[k] = loss.item()

        out[split] = losses.mean().item()

    # Switch back to training mode
    model.train()

    return out

In [64]:
learning_rate = 3e-4

max_iters = 5000
eval_interval = 500
eval_iters = 100

In [65]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

for iter in range(max_iters):

    # Evaluate train and validation loss
    if iter % eval_interval == 0:

        losses = estimate_loss()

        print(
            f"Step {iter}: "
            f"Train Loss = {losses['train']:.4f}, "
            f"Val Loss = {losses['val']:.4f}"
        )

    # Get training batch
    xb, yb = get_batch("train")

    # Forward pass
    logits, loss = model(xb, yb)

    # Clear old gradients
    optimizer.zero_grad(
        set_to_none=True
    )

    # Backpropagation
    loss.backward()

    # Update parameters
    optimizer.step()

Step 0: Train Loss = 7.2290, Val Loss = 7.2336
Step 500: Train Loss = 3.2355, Val Loss = 3.2206
Step 1000: Train Loss = 2.8380, Val Loss = 2.8340
Step 1500: Train Loss = 2.4505, Val Loss = 2.4655
Step 2000: Train Loss = 2.2470, Val Loss = 2.2820
Step 2500: Train Loss = 2.1142, Val Loss = 2.1665
Step 3000: Train Loss = 2.0254, Val Loss = 2.0826
Step 3500: Train Loss = 1.9481, Val Loss = 2.0289
Step 4000: Train Loss = 1.8737, Val Loss = 1.9750
Step 4500: Train Loss = 1.8289, Val Loss = 1.9457


In [73]:
@torch.no_grad()
def generate(self, idx, max_new_tokens, temperature=1.0):

    for _ in range(max_new_tokens):

        # Keep only the last block_size tokens
        idx_cond = idx[:, -block_size:]

        # Forward pass
        logits, _ = self(idx_cond)

        # Take predictions from the last position
        logits = logits[:, -1, :]

        # Apply temperature
        logits = logits / temperature

        # Convert logits to probabilities
        probs = F.softmax(
            logits,
            dim=-1
        )

        # Sample the next token
        idx_next = torch.multinomial(
            probs,
            num_samples=1
        )

        # Add predicted token
        idx = torch.cat(
            (idx, idx_next),
            dim=1
        )

    return idx

In [74]:
import types

model.generate = types.MethodType(
    generate,
    model
)

In [75]:
print(hasattr(model, "generate"))

True


In [79]:
model.eval()

# Start with a simple prompt
prompt = "One day"

# Encode prompt using your BPE tokenizer
context = torch.tensor(
    [tokenizer.encode(prompt)],
    dtype=torch.long,
    device=device
)

# Generate new tokens
generated_ids = model.generate(
    context,
    max_new_tokens=300,
    temperature=0.7
)

# Convert generated IDs back to text
generated_text = tokenizer.decode(
    generated_ids[0].tolist()
)

print(generated_text)

One day, a little girl named Daisy. She wanted to climbed to the park place to be better.

So the pirate noticed something to the provine that it was beautiful hard. She was thought of no diary just about the puppy was so happy to write it.
One day, a little girl called Claire and she had a walk outside playing with her toy. She was a little girl called she was very excited to dry one.
Once upon a time, there was a little girl called Jen old man. She loved to play with her friends in the forest. One day, she got the penny and she saw a big quiet. 

Lucy was so excited that she had found a big tree and it was cold. She was so proud of her best to help


In [80]:
final_losses = estimate_loss()

print(
    f"Final Train Loss: {final_losses['train']:.4f}"
)

print(
    f"Final Val Loss: {final_losses['val']:.4f}"
)

Final Train Loss: 1.7727
Final Val Loss: 1.9079
